In [1]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
from tqdm import tqdm
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def create_driver():
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def get_kakao_all_info(driver, url, max_reviews=100):
    # place.map.kakao.com 직접 URL → iframe 없음, 리뷰가 바로 페이지에 있음
    driver.get(url)
    time.sleep(3)

    # desc_review 로딩 대기 - 실패하면 예외 발생 → 재시도
    WebDriverWait(driver, 15).until(
        EC.presence_of_element_located((By.CLASS_NAME, "desc_review"))
    )

    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1.5)

    collected_reviews = []

    while len(collected_reviews) < max_reviews:
        # 각 리뷰 '더보기' 버튼 클릭 (텍스트 전문 펼치기)
        while True:
            clicked = False
            for i in range(len(driver.find_elements(By.CLASS_NAME, "btn_more"))):
                try:
                    btn = driver.find_elements(By.CLASS_NAME, "btn_more")[i]
                    driver.execute_script("arguments[0].scrollIntoView(true);", btn)
                    if btn.text.strip() == "더보기":
                        driver.execute_script("arguments[0].click();", btn)
                        time.sleep(0.5)
                        clicked = True
                        break
                except:
                    continue
            if not clicked:
                break

        # 펼쳐진 리뷰 텍스트 수집
        for el in driver.find_elements(By.CLASS_NAME, "desc_review"):
            txt = el.text.strip().replace("\n", " ").replace("접기", "").strip()
            if txt and txt not in collected_reviews:
                collected_reviews.append(txt)
            if len(collected_reviews) >= max_reviews:
                break

        if len(collected_reviews) >= max_reviews:
            break

        # '후기 더보기' 버튼 클릭 (다음 배치 로드) - 여러 방법 시도
        next_loaded = False

        # 방법 1: link_more 클래스
        try:
            for btn in driver.find_elements(By.CLASS_NAME, "link_more"):
                if "후기 더보기" in btn.text:
                    driver.execute_script("arguments[0].scrollIntoView(true);", btn)
                    driver.execute_script("arguments[0].click();", btn)
                    time.sleep(2)
                    next_loaded = True
                    break
        except:
            pass

        # 방법 2: 텍스트로 버튼 직접 탐색
        if not next_loaded:
            try:
                btns = driver.find_elements(By.XPATH, "//*[contains(text(), '후기 더보기')]")
                if btns:
                    driver.execute_script("arguments[0].scrollIntoView(true);", btns[0])
                    driver.execute_script("arguments[0].click();", btns[0])
                    time.sleep(2)
                    next_loaded = True
            except:
                pass

        if not next_loaded:
            break  # 버튼 없음 → 더 이상 리뷰 없음

        # 클릭 후 새 리뷰가 실제로 추가됐는지 확인 (무한루프 방지)
        time.sleep(0.5)
        new_elements = driver.find_elements(By.CLASS_NAME, "desc_review")
        if not new_elements:
            break
        new_count = sum(
            1 for el in new_elements
            if el.text.strip().replace("\n", " ").replace("접기", "").strip()
            and el.text.strip().replace("\n", " ").replace("접기", "").strip() not in collected_reviews
        )
        if new_count == 0:
            break

    return {'reviews': collected_reviews}

def get_kakao_with_retry(driver, url, max_reviews=100, retries=3):
    for attempt in range(retries):
        try:
            return get_kakao_all_info(driver, url, max_reviews)
        except Exception as e:
            if attempt < retries - 1:
                print(f"\n[재시도 {attempt+1}] {url}")
                time.sleep(3)
            else:
                print(f"\n[최종 실패] {url}")
                return {'reviews': []}

# --- 메인 실행부 ---
df = pd.read_csv('한국관광공사_제주도_전처리.csv', encoding='cp949')

# 이미 처리된 항목 건너뛰기 (중단 후 재시작 지원)
try:
    backup_df = pd.read_csv('crawling_backup.csv', encoding='utf-8-sig')
    done_mask = backup_df['reviews_text'].notna() & (backup_df['reviews_text'] != '')
    start_idx = int(done_mask.sum())
    reviews_list = backup_df['reviews_text'].fillna('').tolist()[:start_idx]
    print(f"이전 백업 발견: {start_idx}개 완료, {start_idx}번부터 재시작")
except:
    start_idx = 0
    reviews_list = []
    print("처음부터 시작")

driver = create_driver()
fail_indices = []

print(f"총 {len(df)}개 중 {len(df) - start_idx}개 크롤링 시작 (리뷰 최대 100개/장소)")

for idx, row in tqdm(df.iloc[start_idx:].iterrows(), total=len(df) - start_idx):
    if (idx - start_idx) > 0 and (idx - start_idx) % 150 == 0:
        driver.quit()
        time.sleep(2)
        driver = create_driver()

    data = get_kakao_with_retry(driver, row['place_url'], max_reviews=100)
    review_text = " | ".join(data['reviews'])
    reviews_list.append(review_text)

    if not data['reviews']:
        fail_indices.append(idx)

    if (idx + 1) % 50 == 0:
        temp_df = df.copy()
        temp_df['reviews_text'] = reviews_list + [''] * (len(df) - len(reviews_list))
        temp_df.to_csv('crawling_backup.csv', index=False, encoding='utf-8-sig')
        print(f"\n[backup] {idx + 1}번째 완료 → crawling_backup.csv 저장")

driver.quit()

df['reviews_text'] = reviews_list
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
df.to_csv('jeju_crawling_100.csv', index=False, encoding='utf-8-sig')

# 실패 목록 저장
if fail_indices:
    fail_df = df.iloc[fail_indices]
    fail_df.to_csv('crawling_fail_list.csv', index=False, encoding='utf-8-sig')
    print(f"\n실패한 장소 {len(fail_indices)}개 → crawling_fail_list.csv 저장")

# 결과 요약
review_counts = df['reviews_text'].apply(lambda x: len(str(x).split(' | ')) if pd.notna(x) and str(x).strip() else 0)
print(f"\n=== 크롤링 완료 ===")
print(f"총 장소: {len(df)}개")
print(f"리뷰 0개: {(review_counts == 0).sum()}개")
print(f"리뷰 1~49개: {((review_counts > 0) & (review_counts < 50)).sum()}개")
print(f"리뷰 50~99개: {((review_counts >= 50) & (review_counts < 100)).sum()}개")
print(f"리뷰 100개: {(review_counts == 100).sum()}개")
print(f"평균 리뷰 수: {review_counts.mean():.1f}개")
print("완료! → 'jeju_crawling_100.csv' 저장")

처음부터 시작
총 1054개 중 1054개 크롤링 시작 (리뷰 최대 100개/장소)


  5%|▍         | 50/1054 [08:06<2:49:52, 10.15s/it]


[backup] 50번째 완료 → crawling_backup.csv 저장


  9%|▉         | 100/1054 [16:12<2:23:39,  9.04s/it]


[backup] 100번째 완료 → crawling_backup.csv 저장


...


[backup] 1000번째 완료 → crawling_backup.csv 저장


100%|█████████▉| 1050/1054 [2:43:50<00:26,  6.71s/it]


[backup] 1050번째 완료 → crawling_backup.csv 저장


100%|██████████| 1054/1054 [2:44:43<00:00,  9.38s/it]



실패한 장소 1개 → crawling_fail_list.csv 저장

=== 크롤링 완료 ===
총 장소: 1054개
리뷰 0개: 1개
리뷰 1~49개: 1053개
리뷰 50~99개: 0개
리뷰 100개: 0개
평균 리뷰 수: 17.9개
완료! → 'jeju_crawling_100.csv' 저장
